In [ ]:
!nvidia-smi
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

Sun Jan 25 18:18:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             33W /   70W |    2466MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -U transformers peft accelerate sentencepiece
!pip install llama-cpp-python -q

In [ ]:
!pip install bitsandbytes

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
LORA_PATH = "/content/adapters"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, LORA_PATH)

model = model.merge_and_unload()

model.save_pretrained("./merged_model")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained("./merged_model")


('./merged_model/tokenizer_config.json',
 './merged_model/special_tokens_map.json',
 './merged_model/chat_template.jinja',
 './merged_model/tokenizer.model',
 './merged_model/added_tokens.json',
 './merged_model/tokenizer.json')

In [ ]:
!ls /content/adapters


adapter_config.json	   checkpoint-45  special_tokens_map.json
adapter_model.safetensors  checkpoint-90  tokenizer_config.json
chat_template.jinja	   hr_adapter	  tokenizer.json
checkpoint-135		   README.md	  tokenizer.model


In [5]:
!git clone https://github.com/ggerganov/llama.cpp


Cloning into 'llama.cpp'...
remote: Enumerating objects: 77095, done.
remote: Counting objects: 100% (284/284), done.
remote: Compressing objects: 100% (158/158), done.
remote: Total 77095 (delta 222), reused 126 (delta 126), pack-reused 76811 (from 3)
Receiving objects: 100% (77095/77095), 284.08 MiB | 16.15 MiB/s, done.
Resolving deltas: 100% (55723/55723), done.
Updating files: 100% (2146/2146), done.


In [6]:
%cd ./llama.cpp


/content/llama.cpp/llama.cpp


In [9]:
!cmake -B build
!cmake --build build --config Release -j
# cmake --build . --parallel 1


CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.9.5
-- ggml commit:  0440bfd16
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: common
-- Configuring done (0.6s)
-- Generating done (0.4s)
-- Build files have been written to: /content/llama.cpp/llama.cpp/build
[  0%] Built target build_info
[  1%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  1%] Built target sha256
[  2%] Built target sha1
[  2%] Built target llama-gemma3-cli
[  3%] Built target llama-minicpmv-cli
[  3%] Built target xxhash
[  3%] Built target llama-llava-cli
[  4%] Built target llama-qwen2vl-cli
[  7%] Built target ggml-base
[ 10%] Built target ggml-cpu
[ 10%] Built target ggml
[ 10%] Built 

In [20]:
cd /content/llama.cpp


/content/llama.cpp


In [26]:
!ls /content


adapters   merged_model  tiny_llama_finetuned.zip  val.jsonl
llama.cpp  sample_data	 train.jsonl


In [27]:
!find /content -maxdepth 3 -type d | grep merged


/content/merged_model


In [29]:
!python convert_hf_to_gguf.py /content/merged_model \
  --outfile /content/model-q8_0.gguf \
  --outtype q8_0


INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> Q8_0, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> Q8_0, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.

In [21]:
ls | grep convert


convert_hf_to_gguf.py*
convert_hf_to_gguf_update.py*
convert_llama_ggml_to_gguf.py*
convert_lora_to_gguf.py*


In [65]:
!ls llama.cpp | grep convert


convert_hf_to_gguf.py
convert_hf_to_gguf_update.py
convert_llama_ggml_to_gguf.py
convert_lora_to_gguf.py


In [67]:
!python llama.cpp/llama.cpp/convert_hf_to_gguf.py ./merged_model \
  --outfile model-q8_0.gguf \
  --outtype q8_0

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> Q8_0, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> Q8_0, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.

In [69]:
!python llama.cpp/llama.cpp/convert_hf_to_gguf.py ./merged_model \
  --outfile model-bf16.gguf \
  --outtype bf16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> BF16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> BF16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> BF16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> BF16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> BF16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> BF16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.

In [70]:
!python llama.cpp/llama.cpp/convert_hf_to_gguf.py ./merged_model \
  --outfile model-f16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_o

In [71]:
!ls


adapters      model-bf16.gguf  sample_data		 val.jsonl
llama.cpp     model-f16.gguf   tiny_llama_finetuned.zip
merged_model  model-q8_0.gguf  train.jsonl


In [72]:
%cd /content/llama.cpp

/content/llama.cpp


In [73]:
%cd build

/content/llama.cpp/build


In [74]:
cd bin

/content/llama.cpp/build/bin


In [75]:
!ls /content/llama.cpp/build


bin		       CTestTestfile.cmake    llama.pc		   tests
CMakeCache.txt	       DartConfiguration.tcl  llama-version.cmake  tools
CMakeFiles	       examples		      Makefile		   vendor
cmake_install.cmake    ggml		      pocs
common		       license.cpp	      src
compile_commands.json  llama-config.cmake     Testing


In [76]:
!ls /content/llama.cpp/build/bin


libggml-base.so        libggml-cpu.so.0.9.5  llama-gguf
libggml-base.so.0      libggml.so	     llama-llava-cli
libggml-base.so.0.9.5  libggml.so.0	     llama-minicpmv-cli
libggml-cpu.so	       libggml.so.0.9.5      llama-qwen2vl-cli
libggml-cpu.so.0       llama-gemma3-cli


In [77]:
ls

libggml-base.so@        libggml-cpu.so.0.9.5*  llama-gguf*
libggml-base.so.0@      libggml.so@            llama-llava-cli*
libggml-base.so.0.9.5*  libggml.so.0@          llama-minicpmv-cli*
libggml-cpu.so@         libggml.so.0.9.5*      llama-qwen2vl-cli*
libggml-cpu.so.0@       llama-gemma3-cli*


In [78]:
%cd ..
%cd ..
%cd ..

/content/llama.cpp/build
/content/llama.cpp
/content


In [83]:
!ls -lh ./model-f16.gguf


-rw-r--r-- 1 root root 2.1G Jan 25 19:13 ./model-f16.gguf


In [95]:
!./llama.cpp/llama.cpp/build/bin/llama-quantize model-bf16.gguf model-q4_0.gguf q4_0


main: build = 7835 (0440bfd16)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'model-bf16.gguf' to 'model-q4_0.gguf' as Q4_0
llama_model_loader: direct I/O is enabled, disabling mmap
llama_model_loader: loaded meta data with 32 key-value pairs and 201 tensors from model-bf16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32          

In [93]:
!./llama.cpp/llama.cpp/build/bin/llama-quantize \
  model-f16.gguf \
  model-q8_0.gguf \
  q8_0


main: build = 7835 (0440bfd16)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'model-f16.gguf' to 'model-q8_0.gguf' as Q8_0
llama_model_loader: direct I/O is enabled, disabling mmap
llama_model_loader: loaded meta data with 32 key-value pairs and 201 tensors from model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32            

In [96]:
import os

files = {
    "FP16 GGUF": "model-f16.gguf",
    "INT8 GGUF": "model-q8_0.gguf",
    "INT4 GGUF": "model-q4_0.gguf"
}

print(f"{'Format':<15} | {'Size (MB)':>10}")
print("-" * 30)

for name, path in files.items():
    size = os.path.getsize(path) / 1024**2
    print(f"{name:<15} | {size:>10.2f}")

Format          |  Size (MB)
------------------------------
FP16 GGUF       |    2099.05
INT8 GGUF       |    1115.62
INT4 GGUF       |     607.23


In [97]:
import os
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "./merged_model"
OUT_DIR = "./quantized"
os.makedirs(OUT_DIR, exist_ok=True)

def quantize(bits):
    print(f"\n--- INT{bits} Quantization ---")

    if bits == 8:
        bnb = BitsAndBytesConfig(load_in_8bit=True)
    else:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        quantization_config=bnb
    )

    save_path = f"{OUT_DIR}/model-int{bits}"
    model.save_pretrained(save_path)

    mem = model.get_memory_footprint() / 1024**2
    print(f"Memory footprint: {mem:.2f} MB")

    del model
    torch.cuda.empty_cache()

quantize(8)
quantize(4)


--- INT8 Quantization ---


Exception ignored in: <function LlamaModel.__del__ at 0x7d9d7415a8e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/llama_cpp/_internals.py", line 86, in __del__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/llama_cpp/_internals.py", line 78, in close
    if self.sampler is not None:
       ^^^^^^^^^^^^
AttributeError: 'LlamaModel' object has no attribute 'sampler'


Memory footprint: 1174.18 MB

--- INT4 Quantization ---
Memory footprint: 712.18 MB


In [88]:
from llama_cpp import Llama

llm = Llama(
    model_path="model-f16.gguf",
    n_ctx=2048,
    chat_format="llama-3",
    repetition_penalty=1.2,
    verbose=False
)

response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "Analyze the HR scenario step by step, clearly explain the HR reasoning, and conclude."},
        {"role": "user", "content": "What are stay bonuses and when are they used?"}
    ],
    temperature=0.1,
    max_tokens=300
)


print(response["choices"][0]["message"]["content"])


Sure, stay bonuses are a type of performance-based compensation that are awarded to employees who meet or exceed their performance targets. They are typically used to reward employees who have demonstrated exceptional performance and have helped the company achieve its goals. In the HR scenario, the company has identified a specific employee, John, who has consistently exceeded his targets and has been recognized for his outstanding performance. As a result, the company has decided to offer John a stay bonus of $10,000. The stay bonus is awarded to John as a reward for his exceptional performance and is intended to recognize his hard work and dedication to the company. The stay bonus is a form of performance-based compensation that is designed to motivate and reward employees who have demonstrated exceptional performance. It is a valuable incentive for employees to continue to perform at a high level and to contribute to the success of the company.


In [98]:
import os

models = {
    "BF16": "./merged_model",
    "INT8 (bnb)": "./quantized/model-int8",
    "INT4 (bnb)": "./quantized/model-int4",
    "GGUF bf16": "model-bf16.gguf",
    "GGUF f16": "model-f16.gguf",
    "GGUF q8_0": "model-q8_0.gguf",
    "GGUF q4_0": "model-q4_0.gguf",
}

def get_size_mb(path):
    if os.path.isfile(path):
        return os.path.getsize(path) / 1024**2
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1024**2

print(f"{'Format':<15} | {'Size (MB)':>10}")
print("-" * 30)

for name, path in models.items():
    print(f"{name:<15} | {get_size_mb(path):>10.2f}")

Format          |  Size (MB)
------------------------------
BF16            |    2102.13
INT8 (bnb)      |    1175.73
INT4 (bnb)      |     727.13
GGUF bf16       |    2099.05
GGUF f16        |    2099.05
GGUF q8_0       |    1115.62
GGUF q4_0       |     607.23


In [99]:
from llama_cpp import Llama
import time

def benchmark_gguf(path, label, prompt):
    llm = Llama(
        model_path=path,
        n_ctx=2048,
        n_threads=8,
        verbose=False
    )

    start = time.time()
    llm(prompt, max_tokens=128, echo= False)
    end = time.time()

    tps = 128 / (end - start)
    print(f"{label}: {tps:.2f} tokens/sec")
p1 = "What is HR digitalization?"
p2 = "Why is succession planning important?"
benchmark_gguf("model-q8_0.gguf", "GGUF q8_0", p1)
benchmark_gguf("model-q4_0.gguf", "GGUF q4_0", p2)
benchmark_gguf("model-f16.gguf", "GGUF f16", p1)
benchmark_gguf("model-bf16.gguf", "GGUF bf16", p1)


GGUF q8_0: 5.80 tokens/sec
GGUF q4_0: 361.98 tokens/sec
GGUF f16: 15.54 tokens/sec
GGUF bf16: 11.38 tokens/sec
